# Data-type scaling

Six geometry types benchmarked at the same target vertex count
(`N = 50_000`, ± a few %).  Each type is run in two regimes:

- **representative** — natural per-type layout (polylines as locally
  coherent random walks, mesh as a triangulated grid, lines / graph /
  skeleton with scattered positions);
- **scattered** — every vertex drawn independently uniformly across
  the volume, with topology unchanged.  Forces near-maximal cross-chunk
  traffic for types whose representative layout is spatially coherent.

The two regimes isolate the two cost components: per-vertex / per-object
overhead intrinsic to the write/read path, and the cross-chunk cost
driven by spatial distribution.

Each timing is averaged over **`N_RUNS = 10` runs**; bars show the mean
with **95% confidence interval** error bars (Student's t, df=9).
The results table reports the structural metrics (`vertex_count`,
`object_count`, `cross_chunk_count`, `size_MB`) so residual differences
between types can be attributed to specific drivers.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.types.lines import write_lines, read_lines
from zarr_vectors.types.polylines import write_polylines, read_polylines
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.types.meshes import write_mesh, read_mesh

N = 50_000
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0
rng = np.random.default_rng(SEED)

## 2 · Synthetic generators (representative + scattered)

In [ ]:
# Representative — natural per-type layout, ~N total vertices.
def _rep_points():
    return rng.uniform(0, 1000, (N, 3)).astype(np.float32)


def _rep_lines():
    # N/2 segments, two random endpoints each → ~N total vertices.
    return rng.uniform(0, 1000, (N // 2, 2, 3)).astype(np.float32)


def _rep_polylines():
    # ~N total vertices spread across short, locally coherent walks.
    counts = rng.integers(8, 16, size=N // 12)
    out = []
    for c in counts:
        steps = rng.normal(0, 5, (c, 3))
        start = rng.uniform(0, 1000, 3)
        out.append((start + steps.cumsum(axis=0)).astype(np.float32))
    return out


def _rep_graph(is_tree=False):
    positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)
    if is_tree:
        parents = rng.integers(0, np.arange(1, N))
        edges = np.stack([np.arange(1, N), parents], axis=1).astype(np.int64)
    else:
        src = rng.integers(0, N, size=3 * N // 2)
        dst = rng.integers(0, N, size=3 * N // 2)
        mask = src != dst
        edges = np.stack([src[mask], dst[mask]], axis=1).astype(np.int64)
    return positions, edges


def _rep_mesh():
    side = int(np.sqrt(N))
    xs, ys = np.meshgrid(np.linspace(0, 1000, side), np.linspace(0, 1000, side))
    zs = rng.uniform(0, 100, (side, side))
    verts = np.stack([xs, ys, zs], axis=-1).reshape(-1, 3).astype(np.float32)
    i = np.arange(side - 1); j = np.arange(side - 1)
    ii, jj = np.meshgrid(i, j, indexing='ij')
    a = (ii * side + jj).ravel(); b = a + 1; c = a + side; d = c + 1
    faces = np.concatenate([
        np.stack([a, b, c], axis=1),
        np.stack([b, d, c], axis=1),
    ]).astype(np.int64)
    return verts, faces


# Scattered — same topology, every vertex independently uniform in [0, 1000]^3.
# Types whose representative is already uniform reuse the rep generator.
_scat_points = _rep_points
_scat_lines  = _rep_lines
_scat_graph  = _rep_graph


def _scat_polylines():
    counts = rng.integers(8, 16, size=N // 12)
    return [
        rng.uniform(0, 1000, (c, 3)).astype(np.float32)
        for c in counts
    ]


def _scat_mesh():
    # Same grid faces, but vertex positions are uniformly random.
    side = int(np.sqrt(N))
    n_verts = side * side
    verts = rng.uniform(0, 1000, (n_verts, 3)).astype(np.float32)
    i = np.arange(side - 1); j = np.arange(side - 1)
    ii, jj = np.meshgrid(i, j, indexing='ij')
    a = (ii * side + jj).ravel(); b = a + 1; c = a + side; d = c + 1
    faces = np.concatenate([
        np.stack([a, b, c], axis=1),
        np.stack([b, d, c], axis=1),
    ]).astype(np.int64)
    return verts, faces

## 3 · Run the sweep

In [ ]:
def _extract_stats(name, r):
    """Normalize each write_*'s return dict into (vertex_count, object_count, cross_chunk_count)."""
    if name == 'points':
        return r['vertex_count'], r['object_count'], 0
    if name == 'lines':
        return r['line_count'] * 2, r['line_count'], r['cross_chunk_count']
    if name == 'polylines':
        return r['vertex_count'], r['polyline_count'], r['cross_chunk_link_count']
    if name in ('graph', 'skeleton'):
        return r['node_count'], r['node_count'], r['cross_edge_count']
    if name == 'mesh':
        return r['vertex_count'], r['face_count'], r['cross_face_count']
    raise ValueError(name)


def bench_points(gen):
    pts = gen()
    store = _new_store('points')
    tw, res = _time(write_points, store, pts, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _   = _time(read_points, store)
    return tw, tr, res, _store_bytes(store), store

def bench_lines(gen):
    eps = gen()
    store = _new_store('lines')
    tw, res = _time(write_lines, store, eps, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _   = _time(read_lines, store)
    return tw, tr, res, _store_bytes(store), store

def bench_polylines(gen):
    plys = gen()
    store = _new_store('polylines')
    tw, res = _time(write_polylines, store, plys, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _   = _time(read_polylines, store)
    return tw, tr, res, _store_bytes(store), store

def bench_graph(gen, kind):
    pos, edges = gen(is_tree=(kind == 'skeleton'))
    store = _new_store(kind)
    tw, res = _time(
        write_graph, store, pos, edges,
        chunk_shape=CHUNK, bin_shape=BIN, kind=kind,
    )
    tr, _ = _time(read_graph, store)
    return tw, tr, res, _store_bytes(store), store

def bench_mesh(gen):
    verts, faces = gen()
    store = _new_store('mesh')
    tw, res = _time(write_mesh, store, verts, faces, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _   = _time(read_mesh, store)
    return tw, tr, res, _store_bytes(store), store


REGIMES = {
    'representative': {
        'points':    lambda: bench_points(_rep_points),
        'lines':     lambda: bench_lines(_rep_lines),
        'polylines': lambda: bench_polylines(_rep_polylines),
        'graph':     lambda: bench_graph(_rep_graph, kind='graph'),
        'skeleton':  lambda: bench_graph(_rep_graph, kind='skeleton'),
        'mesh':      lambda: bench_mesh(_rep_mesh),
    },
    'scattered': {
        'points':    lambda: bench_points(_scat_points),
        'lines':     lambda: bench_lines(_scat_lines),
        'polylines': lambda: bench_polylines(_scat_polylines),
        'graph':     lambda: bench_graph(_scat_graph, kind='graph'),
        'skeleton':  lambda: bench_graph(_scat_graph, kind='skeleton'),
        'mesh':      lambda: bench_mesh(_scat_mesh),
    },
}
TYPES = ['points', 'lines', 'polylines', 'graph', 'skeleton', 'mesh']

raw    = {r: {name: {'write_s': [], 'read_s': []} for name in TYPES} for r in REGIMES}
struct = {r: {} for r in REGIMES}
sizes  = {r: {} for r in REGIMES}

for regime, fns in REGIMES.items():
    for name in TYPES:
        for run in range(N_RUNS):
            tw, tr, res, nbytes, store = fns[name]()
            raw[regime][name]['write_s'].append(tw)
            raw[regime][name]['read_s'].append(tr)
            if run == 0:
                struct[regime][name] = _extract_stats(name, res)
                sizes[regime][name]  = nbytes
            shutil.rmtree(Path(store).parent, ignore_errors=True)

# Tidy long-form df: one row per (regime, type, op).
rows = []
for regime in REGIMES:
    for name in TYPES:
        v_count, o_count, cc_count = struct[regime][name]
        for op in ('write_s', 'read_s'):
            mean, hw = _mean_ci95(raw[regime][name][op])
            rows.append({
                'regime':            regime,
                'type':              name,
                'op':                op.replace('_s', ''),
                'mean_s':            round(mean, 4),
                'ci_hw':             round(hw, 4),
                'vertex_count':      int(v_count),
                'object_count':      int(o_count),
                'cross_chunk_count': int(cc_count),
                'size_MB':           round(sizes[regime][name] / 1e6, 2),
            })

df = pd.DataFrame(rows)

## 4 · Results

In [ ]:
df

## 5 · Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
x = np.arange(len(TYPES))
w = 0.36

for ax, regime in zip(axes, list(REGIMES)):
    sub   = df[df['regime'] == regime]
    write = sub[sub['op'] == 'write'].set_index('type').loc[TYPES]
    read  = sub[sub['op'] == 'read' ].set_index('type').loc[TYPES]
    ax.bar(x - w/2, write['mean_s'], width=w, yerr=write['ci_hw'],
           capsize=3, label='write')
    ax.bar(x + w/2, read['mean_s'],  width=w, yerr=read['ci_hw'],
           capsize=3, label='read')
    ax.set_xticks(x)
    ax.set_xticklabels(TYPES, rotation=20)
    ax.set_ylabel('seconds')
    ax.set_title(regime)
    ax.grid(True, axis='y', alpha=0.3)

axes[0].legend()
fig.suptitle(
    f'Write / read time per type '
    f'(N ≈ {N:,}, {N_RUNS} runs, 95% CI)'
)
fig.tight_layout()
plt.show()